In [1]:
from panzer.exchanges.binance.public import BinancePublicClient

## Client

In [2]:
client = BinancePublicClient(market="um", safety_ratio=0.5)

2025-11-23 11:39:55     INFO [panzer.binance.um] Market=um REQUEST_WEIGHT limit: RateLimit(rate_limit_type='REQUEST_WEIGHT', interval='MINUTE', interval_num=1, limit=2400)
2025-11-23 11:39:55     INFO [panzer.binance.um] BinancePublicClient inicializado para market=um, base_url=https://fapi.binance.com, safety_ratio=0.5


In [3]:
client = BinancePublicClient(market="cm", safety_ratio=0.5)

2025-11-23 11:39:55     INFO [panzer.binance.cm] Market=cm REQUEST_WEIGHT limit: RateLimit(rate_limit_type='REQUEST_WEIGHT', interval='MINUTE', interval_num=1, limit=2400)
2025-11-23 11:39:55     INFO [panzer.binance.cm] BinancePublicClient inicializado para market=cm, base_url=https://dapi.binance.com, safety_ratio=0.5


In [4]:
client = BinancePublicClient(market="spot", safety_ratio=0.5)

2025-11-23 11:39:56     INFO [panzer.binance.spot] Market=spot REQUEST_WEIGHT limit: RateLimit(rate_limit_type='REQUEST_WEIGHT', interval='MINUTE', interval_num=1, limit=6000)
2025-11-23 11:39:56     INFO [panzer.binance.spot] BinancePublicClient inicializado para market=spot, base_url=https://api.binance.com, safety_ratio=0.5


## Limter test

In [5]:
N = 20
results = []

In [6]:
from time import time
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL

# relaja las condiciones para agilizar pruebas siguientes
limiter = BinanceFixedWindowLimiter(max_per_minute=60,   # límite teórico de 6
                                    safety_ratio=0.9,   # efectivo = 3 req/min
)

for i in range(8):
    t0 = time()
    data, headers = binance_public_get(
        base_url=BINANCE_SPOT_BASE_URL,
        endpoint="/api/v3/time",
        params=None,
        limiter=limiter,
        weight=1,
        timeout=5,
    )
    t1 = time()
    elapsed = t1 - t0
    print(
        f"[i={i:02d}] elapsed={elapsed:.3f}s | "
        f"used_local={limiter.used_local} | "
        f"server_used={limiter.last_server_used}"
    )


[i=00] elapsed=0.256s | used_local=25 | server_used=25
[i=01] elapsed=0.258s | used_local=26 | server_used=26
[i=02] elapsed=0.254s | used_local=27 | server_used=27
[i=03] elapsed=0.255s | used_local=28 | server_used=28
[i=04] elapsed=0.254s | used_local=29 | server_used=29
[i=05] elapsed=0.256s | used_local=30 | server_used=30
[i=06] elapsed=0.261s | used_local=31 | server_used=31
[i=07] elapsed=0.256s | used_local=32 | server_used=32


## Test agresivo con concurrencia

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from panzer.rate_limit.binance_fixed import BinanceFixedWindowLimiter
from panzer.http.client import binance_public_get, BINANCE_SPOT_BASE_URL

limiter = BinanceFixedWindowLimiter(
    max_per_minute=6,
    safety_ratio=0.5,  # effective_limit = 3
)

def worker(idx: int):
    t0 = time()
    data, headers = binance_public_get(
        base_url=BINANCE_SPOT_BASE_URL,
        endpoint="/api/v3/time",
        params=None,
        limiter=limiter,
        weight=1,
        timeout=5,
    )
    print(f"worker {idx} done: data={data}\nheaders={headers}")
    t1 = time()
    elapsed = t1 - t0
    return idx, elapsed, limiter.used_local, limiter.last_server_used

N = 10
with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(worker, i) for i in range(N)]

for fut in as_completed(futures):
    idx, elapsed, used_local, server_used = fut.result()
    print(
        f"[idx={idx:02d}] elapsed={elapsed:.3f}s | "
        f"used_local={used_local} | server_used={server_used}"
    )


2025-11-23 11:39:58  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 1.39 segundos.
2025-11-23 11:39:58  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 1.39 segundos.
2025-11-23 11:39:58  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=33, weight=1, effective_limit=3). Durmiendo 1.10 segundos.
2025-11-23 11:39:58  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=35, weight=1, effective_limit=3). Durmiendo 1.10 segundos.
2025-11-23 11:39:58  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=35, weight=1, effective_limit=3). Durmiendo 1.10 segundos.


worker 0 done: data={'serverTime': 1763894398788}
headers={'Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '28', 'Connection': 'keep-alive', 'Date': 'Sun, 23 Nov 2025 10:39:58 GMT', 'Access-Control-Allow-Methods': 'GET, HEAD, OPTIONS', 'Server': 'nginx', 'x-mbx-uuid': 'b8644a6c-09a0-486b-b1b0-a2265e40f8cd', 'x-mbx-used-weight': '33', 'x-mbx-used-weight-1m': '33', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-Policy': "default-src 'self'", 'X-WebKit-CSP': "default-src 'self'", 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'Access-Control-Allow-Origin': '*', 'X-Cache': 'Miss from cloudfront', 'Via': '1.1 9793e90da776681e20ee7a8a27bdff7c.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'MAD56-P3', 'X-Amz-Cf-Id': '4KJ70n1fAmLUiqzQL

2025-11-23 11:40:00  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 60.00 segundos.
2025-11-23 11:40:00  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 60.00 segundos.
2025-11-23 11:40:00  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 59.72 segundos.
2025-11-23 11:40:00  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 59.72 segundos.


worker 4 done: data={'serverTime': 1763894400164}
headers={'Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '28', 'Connection': 'keep-alive', 'Date': 'Sun, 23 Nov 2025 10:40:00 GMT', 'Access-Control-Allow-Methods': 'GET, HEAD, OPTIONS', 'Server': 'nginx', 'x-mbx-uuid': 'ef61e2d8-9be3-46c9-86db-0b462ed4ca9d', 'x-mbx-used-weight': '1', 'x-mbx-used-weight-1m': '1', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-Policy': "default-src 'self'", 'X-WebKit-CSP': "default-src 'self'", 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'Access-Control-Allow-Origin': '*', 'X-Cache': 'Miss from cloudfront', 'Via': '1.1 e7cf9a8aaf525a2173517459ff93701e.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'MAD56-P3', 'X-Amz-Cf-Id': 'W8LuMUU1t61AMA0hwT3

2025-11-23 11:41:00  WARNING [panzer.binance_rate_limit] Límite de seguridad alcanzado (used_local=3, weight=1, effective_limit=3). Durmiendo 60.00 segundos.


worker 9 done: data={'serverTime': 1763894460183}
headers={'Content-Type': 'application/json;charset=UTF-8', 'Content-Length': '28', 'Connection': 'keep-alive', 'Date': 'Sun, 23 Nov 2025 10:41:00 GMT', 'Access-Control-Allow-Methods': 'GET, HEAD, OPTIONS', 'Server': 'nginx', 'x-mbx-uuid': 'f4e20410-8623-4bbd-875e-ef61eb65706a', 'x-mbx-used-weight': '1', 'x-mbx-used-weight-1m': '1', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains', 'X-Frame-Options': 'SAMEORIGIN', 'X-Xss-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'Content-Security-Policy': "default-src 'self'", 'X-Content-Security-Policy': "default-src 'self'", 'X-WebKit-CSP': "default-src 'self'", 'Cache-Control': 'no-cache, no-store, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'Access-Control-Allow-Origin': '*', 'X-Cache': 'Miss from cloudfront', 'Via': '1.1 06742a79e1b18af724346d3eb743f3da.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'MAD56-P3', 'X-Amz-Cf-Id': 'eVV3sCHJHrK_GVlu8as